# Probe calibration — recipe-driven

The robot carries a **probe rod** (`gripper_probe_offset`, or `gripper_probe_90` for sideways-facing pockets) in place of the working tool. Recipes are loaded from the project's **`recipes.j2`** — the *same* file `main.py` uses — so the IK params (`base_distance` / `rail_step` / `rail_span` / `left_approach`) that key each recipe's `calibration_name` are identical to production.

For each `clb_*` anchor on a recipe's component, `recipe.calibrate()` probes three fixed pockets (place1 / place2 → rough XY + Z + tilt, place3 → precise XY), fuses them into one pose, maps it back to the anchor, and stores a single raw/corrected point under the recipe's `calibration_name`. Everything is written to **`core.json` in this project folder** — exactly where production reads it.

> Before running: set `_HERE` (next cell) to this project's directory, and create `scene/calibration.j2` (the production layout with the robot carrying a probe rod instead of the working tool).

In [ ]:
import os, sys, importlib, pkgutil
from pathlib import Path

# ── Project folder ──────────────────────────────────────────────
# Set this to THIS project's directory — the one holding recipes.j2 and scene/.
# Calibration writes core.json to Path.cwd(), so we chdir here; that is also
# where the orchestrator runs main.py from, so the file we write is the file
# production reads. A notebook has no reliable __file__, so pin it explicitly.
_HERE = Path("/home/dorna/Downloads/workspace/workspace/projects/apc")
os.chdir(_HERE)
if str(_HERE) not in sys.path:
    sys.path.insert(0, str(_HERE))

# Register this project's local component types before the scene loads — main.py
# does this for you; a notebook must do it by hand. Import every module under
# components/ so their @register(...) decorators run.
comp_dir = _HERE / "components"
if comp_dir.is_dir():
    for m in pkgutil.iter_modules([str(comp_dir)]):
        if not m.name.startswith("_"):
            importlib.import_module(f"components.{m.name}")

from workspace.workspace import Workspace
from workspace.bt.launcher import load_recipes

In [ ]:
# Calibration scene = the production layout, but the robot carries a probe rod
# instead of the working tool (swap the robot's tool to gripper_probe_offset,
# or gripper_probe_90 for sideways pockets). Mirror the production scene split.
SCENE = ["scene/core_500.j2", "scene/calibration.j2"]   # chassis + probe-tool layout
ws = Workspace(config_path=[str(_HERE / p) for p in SCENE], port=5000)
core = ws.components["core"]
print("components:", sorted(ws.components))
print("probe tool:", [n for n in ws.components if "gripper_probe" in n])

## Load the recipes

The same `recipes.j2` `main.py` uses, via `load_recipes` — so each recipe's `calibration_name` (which embeds the IK params) matches production exactly. Recipes whose component is missing from the calibration scene, or that need an absent device, are skipped with a warning; the calibratable motion recipes load.

In [ ]:
# load_recipes reads recipes.j2 (Jinja2 -> YAML) and instantiates each entry,
# exactly as main.py does. No render vars: project recipes.j2 files set their
# own inline (e.g. apc's `{% set sf = 20 %}`), matching main.py's call.
rcp = load_recipes(ws, core, _HERE / "recipes.j2")
for name, r in rcp.items():
    print(f"{name:24s} calibration_name = {getattr(r, 'calibration_name', None)}")

## Choose what to calibrate

Only motion recipes whose component carries `clb_*` anchors can be calibrated — device-only recipes (e.g. a multimeter) can't. Pick the alias to calibrate and set it once below:

```python
rcp["anode"].calibrate()                        # every clb_ anchor on the component
rcp["disc_in_1"].calibrate("clb_0")             # one anchor
rcp["disc_in_1"].calibrate(["clb_0", "clb_1"])  # a subset
```

In [ ]:
# The recipe alias to calibrate (set per project / per run).
CAL = "anode"

## Simulation smoke test

Run in sim first to watch the motions end to end. In sim there is no contact sensor, so each touch is faked at the nominal pocket — that fake lives inside the `gripper_probe_*` component, not the recipe.

In [ ]:
core.simulation(True)

# calibration_targets defaults to None -> auto-detect every clb_ anchor on the
# recipe's component. Pass an anchor name or list to do a subset.
rcp[CAL].calibrate()

In [ ]:
name = rcp[CAL].calibration_name
entries = core.calibration.calibration_data.get(name, [])
print(f"{len(entries)} entries under '{name}'")
print(f"core.json -> {Path.cwd() / 'core.json'}")
for e in entries:
    print("  raw       =", [round(v, 3) for v in e["raw"]])
    print("  corrected =", [round(v, 3) for v in e["corrected"]])

## Real robot

Physically mount the correct probe rod (offset, or 90° for sideways pockets), turn simulation off, confirm the probe input reads the **safe** value (`0` = not touching, `1` = touching), then calibrate.

In [ ]:
# core.simulation(False)
# assert not core._simulation_mode, "core still in simulation"
# print("probe input 7:", core.dorna.input(index=7), "(expect safe = 0)")
# rcp[CAL].calibrate()